In [1]:
import os
import sys

repo_url = "<...>"
repo_name = "dnar"

if not os.path.exists(repo_name):
    !git clone {repo_url}
    print(f"Repository '{repo_name}' cloned successfully.")
else:
    print(f"Repository '{repo_name}' already exists. Skipping cloning.")

repo_path = os.path.abspath(repo_name)
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)
    print(f"'{repo_name}' added to Python path: {repo_path}")
else:
    print(f"'{repo_name}' is already in Python path.")


/bin/bash: -c: line 1: syntax error near unexpected token `newline'
/bin/bash: -c: line 1: `git clone <...>'
Repository 'dnar' cloned successfully.
'dnar' added to Python path: /content/dnar


In [2]:
import numpy as np
import torch

import utils
from configs import base_config
from generate_data import create_dataloader
from models import Dnar
from train import train


ModuleNotFoundError: No module named 'utils'

In [ ]:
# SAT config
config_path = './configs/sat.yaml'
config = base_config.read_config(config_path)
assert config.algorithm == 'sat'

torch.set_default_tensor_type(torch.DoubleTensor)
torch.set_num_threads(5)


In [ ]:
# Train SAT model(s)
num_seeds = 3
start_seed = 40

torch.autograd.set_detect_anomaly(True)
for seed in range(start_seed, start_seed + num_seeds):
    np.random.seed(seed)
    torch.manual_seed(seed)
    train(config, seed)


In [ ]:
# Evaluate SAT model checkpoint
split = 'val'
seed = 100
model_seed = 42
model_suffix = 'last'

np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Dnar(config)
model_path = f"{config.models_directory}/{config.algorithm}_{model_seed}_{model_suffix}"
model.load_state_dict(torch.load(model_path, map_location='cpu'))
model = model.to(device)

dataloader = create_dataloader(config, split, seed=seed, device=device)
with torch.no_grad():
    scores = utils.evaluate(model, dataloader, utils.METRICS[config.output_type])
scores
